In [1]:
pip install pymilvus[milvus_lite]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 34.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.3/315.3 kB 14.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# HuggingFace login (add HF_TOKEN in Kaggle Secrets)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)

In [8]:
import os
import json
from collections import defaultdict
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pymilvus import (
    connections, utility,
    FieldSchema, CollectionSchema, DataType,
    Collection
)

PREDICTIONS_PATH = "/kaggle/input/datasets/tanuadhikari/predictions/predictions.txt"
DB_PATH          = "/kaggle/working/legal_ir.db"

doc_labels = {}
with open(PREDICTIONS_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) != 2:
            continue
        doc_id, labels_str = parts
        doc_labels[doc_id] = labels_str.split(",")

print(f"Loaded labels for {len(doc_labels)} documents")

datasets = load_dataset("Exploration-Lab/IL-TUR", "pcr")
train_data = datasets['train_candidates']

doc_sentences = {}
for sample in train_data:
    doc_sentences[str(sample['id'])] = sample['text']

print(f"Loaded sentences for {len(doc_sentences)} documents")

# Filter Roles
KEEP_ROLES = {
    "Facts",
    "Issues",
    "Ratio of the decision",
}

def filter_sentences(sentences, labels, keep_roles):
    filtered = []
    for sent, label in zip(sentences, labels):
        if label in keep_roles:
            filtered.append(sent.strip())
    return filtered

filtered_docs = {}
skipped = 0
for doc_id, labels in doc_labels.items():
    sents = doc_sentences.get(doc_id)
    if sents is None:
        skipped += 1
        continue
    min_len = min(len(sents), len(labels))
    kept = filter_sentences(sents[:min_len], labels[:min_len], KEEP_ROLES)
    if kept:
        filtered_docs[doc_id] = " ".join(kept)

print(f"Filtered corpus: {len(filtered_docs)} docs | Skipped: {skipped}")

#Load Embedding Model
print("Loading embedding model")
embed_model = SentenceTransformer("Snowflake/snowflake-arctic-embed-l")
DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {DIM}")

# Connect to Milvus Lite
from pymilvus import MilvusClient

client = MilvusClient(DB_PATH)
print("Milvus Lite connected")

COLLECTION_NAME = "legal_corpus"

if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)
    print("Dropped existing collection")

client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=DIM,
    metric_type="L2",
    id_field_name="id",
    vector_field_name="embedding",
    enable_dynamic_field=True,
)
print(f"Collection '{COLLECTION_NAME}' created with dim={DIM}")

# Encode & Insert
BATCH_SIZE = 64
doc_ids   = list(filtered_docs.keys())
doc_texts = [filtered_docs[d] for d in doc_ids]

print(f"Encoding {len(doc_texts)} documents...")

for batch_start in range(0, len(doc_texts), BATCH_SIZE):
    batch_texts = doc_texts[batch_start : batch_start + BATCH_SIZE]
    batch_ids   = doc_ids  [batch_start : batch_start + BATCH_SIZE]

    embeddings = embed_model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
    ).tolist()

    data = [
        {
            "id":        int(bid) if bid.isdigit() else hash(bid) % (2**31),
            "embedding": emb,
            "doc_id":    bid,
            "text":      txt[:4000],
        }
        for bid, emb, txt in zip(batch_ids, embeddings, batch_texts)
    ]

    client.insert(collection_name=COLLECTION_NAME, data=data)

    if (batch_start // BATCH_SIZE) % 10 == 0:
        print(f"  Inserted {min(batch_start + BATCH_SIZE, len(doc_texts))}/{len(doc_texts)}")

print("All documents inserted!")
with open("/kaggle/working/doc_texts.json", "w") as f:
    json.dump(filtered_docs, f)
print("Saved doc_texts.json")

test_id   = doc_ids[0]
test_text = filtered_docs[test_id]
test_vec  = embed_model.encode([test_text], normalize_embeddings=True).tolist()

results = client.search(
    collection_name=COLLECTION_NAME,
    data=test_vec,
    limit=11,
    output_fields=["doc_id"],
)

print(f"\nSanity check — top-5 results for doc {test_id}:")
for hit in results[0]:
    print(f"  doc_id={hit['entity']['doc_id']}  distance={hit['distance']:.4f}")

result = client.get(
    collection_name=COLLECTION_NAME,
    ids=[int("0001451408")],
    output_fields=["doc_id", "text"]
)
print("doc_id :", result[0]["doc_id"])
print("text preview:", result[0]["text"][:300])

Loaded labels for 4320 documents


README.md: 0.00B [00:00, ?B/s]

pcr/train_candidates-00000-of-00001.parq(…):   0%|          | 0.00/77.0M [00:00<?, ?B/s]

pcr/dev_candidates-00000-of-00001.parque(…):   0%|          | 0.00/25.1M [00:00<?, ?B/s]

pcr/test_candidates-00000-of-00001.parqu(…):   0%|          | 0.00/34.2M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Embedding dimension: 1024
Milvus Lite connected
Collection 'legal_corpus' created with dim=1024
Encoding 4056 documents...
  Inserted 64/4056
  Inserted 704/4056
  Inserted 1344/4056
  Inserted 1984/4056
  Inserted 2624/4056
  Inserted 3264/4056
  Inserted 3904/4056
All documents inserted!
Saved doc_texts.json

Sanity check — top-5 results for doc 0001451408:
  doc_id=0001451408  distance=0.0000
  doc_id=0000338648  distance=0.3656
  doc_id=0000723416  distance=0.3774
  doc_id=0001747962  distance=0.3801
  doc_id=0000262260  distance=0.3828
  doc_id=0001806797  distance=0.3895
  doc_id=0000274311  distance=0.3904
  doc_id=0001633734  distance=0.4015
  doc_id=0001898146  distance=0.4052
  doc_id=0000412326  distance=0.4069
  doc_id=0000197199  distance=0.4074
doc_id : 0001451408
text preview: 16. It was next contended that even though proceedings for each assessment year are separate only one notice was given in respect of the assessment years 1973-74 to 1977-78, and hence the notice is

E0331 17:02:01.370551     738 chttp2_transport.cc:1385] unix:/tmp/tmp5gqck8wp_legal_ir.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
